In [71]:
import os
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from pydantic import BaseModel
from langchain_core.messages import BaseMessage
import operator


In [72]:
load_dotenv()

True

In [73]:
api=os.getenv("GROQ_API_KEY")

In [74]:
llm=ChatGroq(api_key=api,model="openai/gpt-oss-20b",
             temperature=0,reasoning_effort="low")

In [75]:
class AgentState(TypedDict):
    task:str
    content_draft:str
    validation_error:list[str]
    retry_count:int
    max_retries:int
    is_valid:bool
    history: Annotated[list[dict], operator.add]

In [76]:
def generator_node(state):
    """
    Generates content based on the task in state.

    First call: writes fresh content from the task instructions.
    Retry call: revises the previous draft using validation_errors as feedback.

    Returns:
        dict: {"content_draft": response.content}
    """

    task=state["task"]
    content_draft=state.get("content_draft","")
    errors=state.get("validation_error","")

    if content_draft:
        prompt=f"""You are a content writer . you previously wrote a draft for this task , but it had issues
        Original Task
        {task}
        
        Previous draft:
        {content_draft}

        Issue found in the previous draft:
        {errors}
Rewrite the content to fix ALL the issues listed above.
Keep everything that was already correct — only fix what's flagged.
Do not introduce new problems while fixing these issues.

"""
    else: 
        prompt =f"""
You are a content writer. Write content based on the following task.

Task:
{task}

Follow the instructions in the task carefully, including any length, tone, or format requirements."""

    response =llm.invoke(prompt)

    return{
        "content_draft":response.content,
    
        "history": [{"node": "generator_node", "content": response.content}]
    }



In [77]:
class Validation(BaseModel):
    is_valid:bool
    
    validation_error:list[str]


In [78]:
def validate_content_node(state):
    """
    Validates the current content draft against quality criteria.

    Checks the draft for issues like length (too long/short), tone mismatch,
    irrelevant content, and formatting problems.

    Returns:
        dict: {"is_valid": bool, "validation_errors": list of found issues}
    """
    content_draft=state.get("content_draft","")
    retry_count = state.get("retry_count", 0)
    task=state["task"]

    

    prompt = f"""
You are a strict content validator.

Original Task:
{task}

Content Draft:
{content_draft}

Validate the draft against ALL requirements in the original task.

Check:
- Exact word count if specified
- Required number of bullet points
- Tone
- Topic/relevance
- Formatting
- Every explicit requirement in the task

Be strict. If even one explicit requirement is not satisfied,
return is_valid=False and clearly explain the issue.

If all requirements are satisfied, return is_valid=True
and an empty validation_error list.
"""

    structured_llm=llm.with_structured_output(Validation,method="json_schema")
    response=structured_llm.invoke(prompt)


   

    result={
        "is_valid":response.is_valid,
       
        "validation_error":response.validation_error,
        "history": [{"node": "validate_content_node", "content": response.validation_error}]
    }
    if not response.is_valid:
        result["retry_count"] = retry_count + 1

    return result


In [79]:
def fallback_node(state):
    """
    Called when max retries are exhausted and content is still invalid.
    Returns the best-available draft with a warning, instead of nothing.
    """
    content_draft = state.get("content_draft", "")
    validation_error = state.get("validation_error", [])

    warning = (
        f"\n\n[Note: This content could not be fully validated after "
        f"multiple attempts. Remaining issues: {validation_error}]"
    )

    return {
        "content_draft": content_draft + warning,
        "history": [{"node": "fallback_node", "content": content_draft + warning}]
    }
    

In [80]:
def route_by_status(state:AgentState) -> str:
    is_valid = state.get("is_valid", False)
    retry_count = state.get("retry_count", 0)
    max_retries = state.get("max_retries", 5)
    
    

    if is_valid:
        return END
    elif retry_count < max_retries:
        return "generator_node"
    else:
         return "fallback_node"

In [81]:
graph=StateGraph(AgentState)

graph.add_node("generator_node",generator_node)
graph.add_node("validate_content_node",validate_content_node)
graph.add_node("fallback_node",fallback_node)


graph.add_edge(START,"generator_node")
graph.add_edge("generator_node","validate_content_node")
#graph.add_conditional_edges("generator_node",route_by_status)
graph.add_conditional_edges("validate_content_node",route_by_status,{
        "generator_node": "generator_node",
        "fallback_node": "fallback_node",
         END: END
    })
#graph.add_edge("generator_node","validate_content_node")


graph.add_edge("fallback_node",END)
app=graph.compile()




In [82]:
initial_state = {
    "task": """
Write a LinkedIn email explaining why production AI systems need
validation and retry mechanisms.

Requirements:
- Professional but human tone
- Around 150 words
- Clear structure
- Avoid unnecessary technical jargon
""",
    "content_draft": "",
    "validation_error": [],
    "retry_count": 0,
    "max_retries": 3,
    "is_valid": False,
    "issue": []
}

final=app.invoke(initial_state)
final

{'task': '\nWrite a LinkedIn email explaining why production AI systems need\nvalidation and retry mechanisms.\n\nRequirements:\n- Professional but human tone\n- Around 150 words\n- Clear structure\n- Avoid unnecessary technical jargon\n',
 'content_draft': '**Subject:** Why Production AI Systems Need Validation & Retry Mechanisms  \n\nHi\u202f[Name],\n\nIn today’s fast‑moving AI landscape, a model that performs well in the lab can still stumble once it’s live. That’s why every deployment must include two essential safeguards:\n\n1. **Validation** – Before any output reaches users, it’s checked against simple business rules or sanity‑checks. This catches outliers, guards against data drift, and keeps the system compliant with regulations.  \n2. **Retry** – When a request fails—whether from a temporary network glitch or a sudden traffic spike—an automated retry keeps the service resilient. Coupled with a back‑off strategy, it prevents cascading failures and preserves a smooth user exper

In [83]:
print("\n--- FINAL OUTPUT ---")
print(final["content_draft"])

print("\n--- VALIDATION ---")
print("Valid:", final["is_valid"])
print("Retries:", final["retry_count"])

if final["validation_error"]:
    print("Errors:", final["validation_error"])
else:
    print("Errors: None")

--- FINAL OUTPUT ---

**Subject:** Why Production AI Systems Need Validation & Retry Mechanisms  

Hi [Name],

In today’s fast‑moving AI landscape, a model that performs well in the lab can still stumble once it’s live. That’s
why every deployment must include two essential safeguards:

1. **Validation** – Before any output reaches users, it’s checked against simple business rules or sanity‑checks. 
This catches outliers, guards against data drift, and keeps the system compliant with regulations.  
2. **Retry** – When a request fails—whether from a temporary network glitch or a sudden traffic spike—an automated 
retry keeps the service resilient. Coupled with a back‑off strategy, it prevents cascading failures and preserves a
smooth user experience.

Together, validation and retry transform a fragile prototype into a reliable, trustworthy service. They’re not 
optional extras; they’re critical for uptime, compliance, and customer confidence.  

Let’s explore how we can embed these mechanisms into your next AI rollout and ensure your solution stays robust, 
compliant, and user‑friendly.

Best regards,  
[Your Name]

--- VALIDATION ---

Valid: True

Retries: 2

Errors: None

In [84]:
initial_state = {
    "task": """
Write a LinkedIn post in exactly 20 words.

Requirements:
- Exactly 20 words
- Include 5 bullet points
- Explain production AI validation in detail
- Use a professional tone
""",
    "content_draft": "",
    "validation_error": [],
    "retry_count": 0,
    "max_retries": 3,
    "is_valid": False,
    "issue": []
}

final=app.invoke(initial_state)
print(final)

{
    'task': '\nWrite a LinkedIn post in exactly 20 words.\n\nRequirements:\n- Exactly 20 words\n- Include 5 bullet 
points\n- Explain production AI validation in detail\n- Use a professional tone\n',
    'content_draft': 'Production AI validation ensures reliability daily.  \n- Data integrity checks  \n- Model 
drift monitoring  \n- Performance benchmarks adherence  \n- Regulatory compliance verification  \n- Continuous 
retraining protocols',
    'validation_error': [],
    'retry_count': 0,
    'max_retries': 3,
    'is_valid': True,
    'history': [
        {
            'node': 'generator_node',
            'content': 'Production AI validation ensures reliability daily.  \n- Data integrity checks  \n- Model 
drift monitoring  \n- Performance benchmarks adherence  \n- Regulatory compliance verification  \n- Continuous 
retraining protocols'
        },
        {'node': 'validate_content_node', 'content': []}
    ]
}

In [85]:


initial_state = {
    "task": """
Create a LinkedIn post about building reliable production-grade AI agents.

Requirements:
- Exactly 100 words
- Start with a strong one-line hook
- Include exactly 3 numbered points
- Each point must explain one practical reliability technique
- Mention validation, retry mechanisms, and observability
- Include one concrete real-world example
- Professional but human tone
- No emojis
- End with a thought-provoking question
- Avoid generic statements and unnecessary technical jargon
""",
    "content_draft": "",
    "validation_error": [],
    "retry_count": 0,
    "max_retries": 3,
    "is_valid": False,
    "issue": []
}
final=app.invoke(initial_state)
final


{'task': '\nCreate a LinkedIn post about building reliable production-grade AI agents.\n\nRequirements:\n- Exactly 100 words\n- Start with a strong one-line hook\n- Include exactly 3 numbered points\n- Each point must explain one practical reliability technique\n- Mention validation, retry mechanisms, and observability\n- Include one concrete real-world example\n- Professional but human tone\n- No emojis\n- End with a thought-provoking question\n- Avoid generic statements and unnecessary technical jargon\n',
 'content_draft': 'When an AI agent fails, the cost is not just a bug—it’s a lost opportunity.\n\n1. Validation – Before deployment, run the agent through a curated test suite that mirrors production data, ensuring every decision path meets business rules.  \n2. Retry mechanisms – Implement exponential back‑off retries for transient failures, coupled with circuit breakers to prevent cascading outages.  \n3. Observability – Log intent, context, and outcome; surface anomalies through

In [86]:
initial_state = {
    "task": """
Create a LinkedIn post about building reliable production-grade AI agents.

Requirements:
- Exactly 100 words
- Start with a strong one-line hook
- Include exactly 3 numbered points
- Each point must explain one practical reliability technique
- Mention validation, retry mechanisms, and observability
- Include one concrete real-world example
- Professional but human tone
- No emojis
- End with a thought-provoking question
- Avoid generic statements and unnecessary technical jargon
""",
    "content_draft": "",
    "validation_error": [],
    "retry_count": 0,
    "max_retries": 3,
    "is_valid": False,
    "issue": []
}
for step in app.stream(initial_state):
    print("\n--- STEP ---")
    print(step)

--- STEP ---

{
    'generator_node': {
        'content_draft': '**Hook:**  \nWhen an AI agent fails, the cost is real—customers lose trust, revenue 
drops, and reputations suffer.\n\n1. **Validation** – Before deployment, run end‑to‑end tests against a curated 
dataset that mirrors production traffic. If a model misclassifies 0.5\u202f% of cases, halt the rollout and 
retrain.  \n2. **Retry mechanisms** – Implement exponential back‑off retries for transient API failures, ensuring 
the agent can recover without manual intervention.  \n3. **Observability** – Continuously log latency, error rates,
and decision confidence; alert on deviations and auto‑scale resources accordingly.\n\nIn a recent e‑commerce 
recommendation engine, these practices cut churn by 12\u202f% and increased conversion by 8\u202f%.  \n\nHow will 
you ensure your AI agents stay reliable under real‑world pressure?',
        'history': [
            {
                'node': 'generator_node',
                'content': '**Hook:**  \nWhen an AI agent fails, the cost is real—customers lose trust, revenue 
drops, and reputations suffer.\n\n1. **Validation** – Before deployment, run end‑to‑end tests against a curated 
dataset that mirrors production traffic. If a model misclassifies 0.5\u202f% of cases, halt the rollout and 
retrain.  \n2. **Retry mechanisms** – Implement exponential back‑off retries for transient API failures, ensuring 
the agent can recover without manual intervention.  \n3. **Observability** – Continuously log latency, error rates,
and decision confidence; alert on deviations and auto‑scale resources accordingly.\n\nIn a recent e‑commerce 
recommendation engine, these practices cut churn by 12\u202f% and increased conversion by 8\u202f%.  \n\nHow will 
you ensure your AI agents stay reliable under real‑world pressure?'
            }
        ]
    }
}

--- STEP ---

{
    'validate_content_node': {
        'is_valid': True,
        'validation_error': [],
        'history': [{'node': 'validate_content_node', 'content': []}]
    }
}